In [ ]:
"""
Module description:

Run script for the GHG emission preparation workflow.
Script prepare point source and area gridded emission input for DALES
Input: CSV files of point source and area emission data of GHGs (1x1 or 5x5 km resolution)
Data can be downloaded here:
https://data.emissieregistratie.nl/export
For mapped data use: Combinatie kaart vk5 NCP  en vk1 Nederland 
For point sources use: Bedrijf

Output: NetCDF files of explicit model input.

The run time of the workflow strongly depends upon the domain size and resolution
Reminder: currently this workflow is suitable to transfer emissions of 1x1km (5x5km too) to the demain resolution and RD coordinates. 
If you some need other or additional setting, feel free to fit scripts of this workflow to your subject/task.

Author: Dr. Arseni Doyennel, VU. IHS EUR
Contact Email: a.doyennel@vu.nl

"""
import os
import xarray as xr

#Import all modules for workflow
from er_data_converter import ERDataConverter
from emission_preparation_setting import spec_name, snaplist, explicit_point, proj_params, grid_params, x0 , y0, x_offset, y_offset, rundir_ruisdael_area_residuals, outputdir_main_ruisdael_area_csv2netc, year_start, cbs_loc_consu_preprocessing, agro_file_name, vaarweg_file, industrial_file_name, ER_excel_to_csv, datadir_point_source_plume_processing, point_ER_excel, map_ER_excel, point_source_harm_file, mapped_data
from point_source_explicit_input_netcdf import Pointsource_input_preparation
from ruisdael_area_residuals import Emission_initial_process_get_residuals
from reproject_to_les import EmissionResampler
from reproject_to_les_snap_7 import Traffic_Disaggregator
from reproject_to_les_snap_3 import SNAP3Refiner
from reproject_to_les_snap_8 import SNAP8Refiner
from reproject_to_les_snap_10 import SNAP10Refiner
from ruisdael_area_total_static import Emission_3Dfield_creation
from create_hourly_emissions_3D import Final_3D_input

if __name__ == "__main__":

    if ER_excel_to_csv:

        converter = ERDataConverter(
            point_excel=(
                datadir_point_source_plume_processing
                + point_ER_excel
            ),
            map_excel=(
                datadir_point_source_plume_processing
                + map_ER_excel
            ),
            point_csv=datadir_point_source_plume_processing+point_source_harm_file,
            map_csv=datadir_point_source_plume_processing+mapped_data,
        )

        converter.convert_all()
        

    if explicit_point:

       PSEP = Pointsource_input_preparation()
       df, df_org = PSEP.identify_emission_categories()
       unassigned_points, df_gapfilled_full = PSEP.prepare_emission_data(df)
       PSEP.prepare_final_input(df_gapfilled_full)

    # Instantiate the class and call the methods:
    emission_init_prep = Emission_initial_process_get_residuals()
    df_point = emission_init_prep.read_point()
    df_point_2 = emission_init_prep.preprocess_point_data(df_point)
    emission_init_prep.save_points_harm(df_point_2)
    field_source, df_map, x, y = emission_init_prep.substract_point_from_map(df_point_2)
    field_subdoelgroep_source_roads = emission_init_prep.populate_road_emissions(df_map, x, y)
    emission_init_prep.save_data(field_source, field_subdoelgroep_source_roads, x, y)
    
    for snap in snaplist:
        if snap == 2:
            
            emission_init_prep.refine_and_save_snap2_data(field_source)
            print(f"Processing SNAP {snap}")
            resampler = EmissionResampler(snap, spec_name, proj_params, grid_params, x0 , y0, x_offset, y_offset, rundir_ruisdael_area_residuals, outputdir_main_ruisdael_area_csv2netc, year_start)
            resampler.run()
            resampler.save_to_netcdf()

        elif snap == 3:  
            print(f"Processing SNAP {snap}")
            resampler = SNAP3Refiner(snap, cbs_loc_consu_preprocessing, industrial_file_name,rundir_ruisdael_area_residuals, 
                                      proj_params, grid_params, x0, y0,x_offset, y_offset, 
                                      spec_name, year_start, outputdir_main_ruisdael_area_csv2netc)
            resampler.run()

        elif snap == 4 and spec_name=='ch4':
            #we just create the nullified file (to place point sources if there are some for snap 4
            infile = os.path.join(
                outputdir_main_ruisdael_area_csv2netc,
                f"HARM_snap_1_all_{spec_name}.nc"
            )
            outfile = os.path.join(
                outputdir_main_ruisdael_area_csv2netc,
                f"HARM_snap_4_all_{spec_name}.nc"
            )

            # Open input dataset
            ds = xr.open_dataset(infile)

            # Make a copy of dataset, zero out the variable
            ds_out = ds.copy()
            if spec_name in ds_out:
                ds_out[spec_name] = ds_out[spec_name] * 0
            else:
                raise KeyError(f"Variable {spec_name} not found in {infile}")

            # Save to new file
            ds_out.to_netcdf(outfile)

            ds.close()
            ds_out.close()
        
        
        elif snap == 7:
            print(f"Processing SNAP {snap}")
            processor = Traffic_Disaggregator()
            processor.run()
            processor.save_to_netcdf()


        elif snap == 8:

            print(f"Processing SNAP {snap} (shipping)")
            resampler = SNAP8Refiner(snap, cbs_loc_consu_preprocessing, vaarweg_file,rundir_ruisdael_area_residuals, 
                                      proj_params, grid_params, x0, y0,x_offset, y_offset, 
                                      spec_name, year_start, outputdir_main_ruisdael_area_csv2netc)
            resampler.run()

        elif snap == 10: 
            
            resampler = SNAP10Refiner(snap, cbs_loc_consu_preprocessing, agro_file_name, rundir_ruisdael_area_residuals, 
                                      proj_params, grid_params, x0, y0,x_offset, y_offset, 
                                      spec_name, year_start, outputdir_main_ruisdael_area_csv2netc)
            resampler.run()
            
        else:
            print(f"Processing SNAP {snap}")
            resampler = EmissionResampler(snap, spec_name, proj_params, grid_params, x0 , y0, x_offset, y_offset, rundir_ruisdael_area_residuals, outputdir_main_ruisdael_area_csv2netc, year_start)
            resampler.run()
            resampler.save_to_netcdf()
    
    E_3D = Emission_3Dfield_creation()
    E_3D.process_emissions()

    emission_final_prep = Final_3D_input()
    emission_final_prep.create_output_directory()
    emission_final_prep.create_settings_file()
    emission_final_prep.create_emissions()